# SleepPy Extraction Review

This notebook reviews first-pass sleep metric extraction from Oura Ring 4 finger, Oura Ring 3 toe, Samsung Watch / SleepWatch, Muse screenshots/PDFs, MindMonitor/Muse sensor-log CSVs, and OSCAR/SleepScope sample screenshots or PDFs.

**Disclaimer:** This is exploratory wellness data analysis only. It is not medical diagnosis, treatment advice, or a replacement for clinician review.


## Workflow

1. Place 1-2 representative files per device under `data/raw/samples/<device>/`.
2. Run the extraction cell to create `data/processed/nightly_summary.csv`, `data/processed/device_observations_long.csv`, and `outputs/extraction_report.md`.
3. Review extracted values, missingness, confidence labels, and trend plots.
4. Correct low-confidence or missing values manually before treating them as analysis inputs.

**After adding or moving raw files, rerun extraction in this notebook by setting `RUN_EXTRACTION = True`, or run `python -m sleeppy.extract --no-legacy-raw` before loading existing processed CSVs. Loading existing CSVs will not discover newly added dated subfolders.**



In [ ]:
from pathlib import Path
import importlib

import matplotlib.pyplot as plt
import pandas as pd

# Reload local modules so an already-running notebook kernel picks up edits from disk.
import sleeppy.schema as sleep_schema
import sleeppy.quality as sleep_quality
import sleeppy.extract as sleep_extract
import sleeppy.extract.common as extract_common
import sleeppy.extract.pipeline as extract_pipeline

importlib.reload(sleep_schema)
importlib.reload(sleep_quality)
importlib.reload(extract_common)
importlib.reload(extract_pipeline)
importlib.reload(sleep_extract)

check_ocr_environment = extract_common.check_ocr_environment
run_sample_extraction = extract_pipeline.run_sample_extraction
confidence_by_device = sleep_quality.confidence_by_device
describe_extraction_outputs = sleep_quality.describe_extraction_outputs
missingness_by_device = sleep_quality.missingness_by_device
observations_to_nightly_summary = sleep_quality.observations_to_nightly_summary
CANONICAL_METRIC_UNITS = sleep_schema.CANONICAL_METRIC_UNITS
OPTIONAL_PLOT_METRICS = sleep_schema.OPTIONAL_PLOT_METRICS
PLOT_METRICS = sleep_schema.PLOT_METRICS
ensure_observations_frame = sleep_schema.ensure_observations_frame
normalize_summary_columns = sleep_schema.normalize_summary_columns

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

MINDMONITOR_DEVICE = "Muse S MindMonitor"
MINDMONITOR_COVERAGE_METRICS = [
    "mindmonitor_session_start_time",
    "mindmonitor_session_end_time",
    "mindmonitor_session_minutes",
    "mindmonitor_valid_eeg_rows",
    "mindmonitor_valid_motion_rows",
    "mindmonitor_valid_ppg_rows",
    "mindmonitor_mean_hr_bpm",
    "mindmonitor_mean_accel_mag",
    "mindmonitor_p95_accel_mag",
    "mindmonitor_stopped_before_morning",
    "mindmonitor_gap_count_gt_5s",
    "mindmonitor_max_gap_seconds",
]

OURA_API_DEVICE = "Oura API"
OURA_SCREENSHOT_DEVICES = ["Oura Ring 4 finger", "Oura Ring 3 toe"]
OURA_API_COMPARISON_METRICS = [
    "sleep_score",
    "total_sleep_minutes",
    "time_in_bed_minutes",
    "sleep_efficiency_pct",
    "rem_minutes",
    "light_minutes",
    "deep_minutes",
    "awake_minutes",
    "avg_hr_bpm",
    "min_hr_bpm",
    "avg_hrv_ms",
    "respiratory_rate_bpm",
]


## OCR Diagnostics

The notebook must have dependencies installed in the active kernel shown below. If `image_ocr_ready` is `False`, run `import sys; !"{sys.executable}" -m pip install -r requirements.txt`, then restart the kernel.

In [ ]:
ocr_status = check_ocr_environment()
display(pd.DataFrame([ocr_status]))

if not ocr_status["image_ocr_ready"]:
    print("Image OCR is not ready in this active notebook kernel. Screenshots will produce zero extracted values until this is fixed.")
    print(ocr_status["notes"])
else:
    print("Image OCR is ready.")

In [ ]:
# Set to True to rerun OCR extraction
RUN_EXTRACTION = False
# Set to True to re-pivot summary from observations
REBUILD_SUMMARY_FROM_OBSERVATIONS = False

nightly_summary_exists = (PROCESSED_DIR / "nightly_summary.csv").exists()

if RUN_EXTRACTION:
    nightly_summary, observations, report_path = run_sample_extraction(
        raw_samples_dir=PROJECT_ROOT / "data" / "raw" / "samples",
        processed_dir=PROCESSED_DIR,
        outputs_dir=OUTPUTS_DIR,
        include_legacy_raw=True,
    )
else:
    print("Loading existing processed data.")
    print("Reminder: set RUN_EXTRACTION=True after adding raw files or dated subfolders.")
    nightly_summary = pd.read_csv(PROCESSED_DIR / "nightly_summary.csv")
    observations = pd.read_csv(PROCESSED_DIR / "device_observations_long.csv")
    report_path = OUTPUTS_DIR / "extraction_report.md"

observations = ensure_observations_frame(observations)

if REBUILD_SUMMARY_FROM_OBSERVATIONS or not nightly_summary_exists:
    if observations.empty:
        nightly_summary = normalize_summary_columns(nightly_summary)
    else:
        nightly_summary = observations_to_nightly_summary(observations)
else:
    nightly_summary = normalize_summary_columns(nightly_summary)

print(f"Night/device rows: {len(nightly_summary)}")
print(f"Extracted values: {len(observations)}")
print(f"Report: {report_path}")
print("Available devices:", sorted(observations["device"].dropna().unique().tolist()))
print("Available observation metrics:", sorted(observations["metric"].dropna().unique().tolist()))
print("Nightly summary columns:", list(nightly_summary.columns))

extraction_diagnostics = describe_extraction_outputs(nightly_summary, observations)


## Extracted Tables

In [ ]:
display(nightly_summary)
display(observations.head(50))

## Missingness and Confidence

In [ ]:
missingness = missingness_by_device(nightly_summary)
confidence = confidence_by_device(observations)

device_counts = (
    observations.groupby("device")
    .size()
    .rename("observation_count")
    .reset_index()
    .sort_values("device")
)
metric_availability = observations.pivot_table(
    index="metric",
    columns="device",
    values="value",
    aggfunc="count",
    fill_value=0,
)

if missingness.empty:
    print("No nightly summary rows available yet.")
else:
    display(missingness.pivot_table(index="metric", columns="device", values="missing_pct"))

if device_counts.empty:
    print("No extracted observations available yet.")
else:
    display(device_counts)

if metric_availability.empty:
    print("No metric availability table available yet.")
else:
    display(metric_availability)

if confidence.empty:
    print("No confidence counts available yet.")
else:
    display(confidence.pivot_table(index="device", columns="confidence", values="count", fill_value=0))


## MindMonitor sensor-log coverage


In [ ]:
mindmonitor_observations = observations[observations["device"].eq(MINDMONITOR_DEVICE)].copy()

if mindmonitor_observations.empty:
    print("No MindMonitor sensor-log observations available.")
else:
    mindmonitor_coverage = mindmonitor_observations[
        mindmonitor_observations["metric"].isin(MINDMONITOR_COVERAGE_METRICS)
    ].copy()
    mindmonitor_coverage["value"] = pd.to_numeric(mindmonitor_coverage["value"], errors="coerce")
    mindmonitor_coverage = mindmonitor_coverage.pivot_table(
        index=["night_date", "source_file"],
        columns="metric",
        values="value",
        aggfunc="first",
    ).reset_index()
    mindmonitor_coverage = mindmonitor_coverage.rename(
        columns={metric: metric.removeprefix("mindmonitor_") for metric in MINDMONITOR_COVERAGE_METRICS}
    )
    display(mindmonitor_coverage)


## Oura API coverage

Oura API rows are optional. Fetch JSON with `python -m sleeppy.api.oura_fetch`, then rerun extraction with `--include-oura-api` or set `RUN_EXTRACTION=True` after cached API files are present.


In [ ]:
oura_api_observations = observations[observations["device"].eq(OURA_API_DEVICE)].copy()

if oura_api_observations.empty:
    print("No Oura API observations available.")
else:
    display(
        oura_api_observations.pivot_table(
            index="metric",
            values="value",
            aggfunc="count",
        ).rename(columns={"value": "observation_count"})
    )

    comparison_obs = observations[
        observations["metric"].isin(OURA_API_COMPARISON_METRICS)
        & observations["device"].isin([OURA_API_DEVICE, *OURA_SCREENSHOT_DEVICES])
    ].copy()
    comparison_obs["value"] = pd.to_numeric(comparison_obs["value"], errors="coerce")
    comparison = comparison_obs.pivot_table(
        index=["night_date", "metric"],
        columns="device",
        values="value",
        aggfunc="first",
    ).reset_index()
    overlap_columns = [column for column in OURA_SCREENSHOT_DEVICES if column in comparison.columns]
    if OURA_API_DEVICE in comparison.columns and overlap_columns:
        display(comparison.dropna(subset=[OURA_API_DEVICE], how="all"))
    else:
        print("No overlapping Oura API/screenshot metrics available for comparison.")


## Trends

In [ ]:
PLOT_SPECS = {
    "total_sleep_minutes": ("Minutes", "Sleep duration"),
    "avg_hrv_ms": ("ms", "Average HRV"),
    "avg_spo2_pct": ("%", "Average SpO2"),
    "cpap_ahi": ("Events/hour", "CPAP AHI"),
}

MINDMONITOR_TREND_SPECS = {
    "mindmonitor_session_minutes": ("Minutes", "MindMonitor session duration"),
    "mindmonitor_mean_hr_bpm": ("bpm", "MindMonitor mean HR"),
    "mindmonitor_mean_accel_mag": ("Magnitude", "MindMonitor mean acceleration magnitude"),
    "mindmonitor_p95_accel_mag": ("Magnitude", "MindMonitor p95 acceleration magnitude"),
}


def available_numeric_metrics(summary_df):
    summary_df = normalize_summary_columns(summary_df)
    metrics = []
    for metric in CANONICAL_METRIC_UNITS:
        if metric in summary_df and pd.to_numeric(summary_df[metric], errors="coerce").notna().any():
            metrics.append(metric)
    return metrics


def plot_metric(summary_df, metric, ylabel, title):
    summary_df = normalize_summary_columns(summary_df)
    if summary_df.empty or metric not in summary_df:
        print(f"No {metric} column available. Available numeric metrics: {available_numeric_metrics(summary_df)}")
        return None

    df = summary_df.copy()
    df["parsed_night_date"] = pd.to_datetime(df["night_date"], errors="coerce")
    df[metric] = pd.to_numeric(df[metric], errors="coerce")
    has_metric_values = df[metric].notna().any()
    plottable = df.dropna(subset=["parsed_night_date", metric])
    if plottable.empty:
        if has_metric_values:
            print(f"{metric} values exist, but none have a parseable night_date for trend plotting.")
        else:
            print(f"No plottable {metric} values available. Available numeric metrics: {available_numeric_metrics(summary_df)}")
        return None

    df = plottable
    if df.empty:
        print(f"No plottable {metric} values available. Available numeric metrics: {available_numeric_metrics(summary_df)}")
        return None

    fig, ax = plt.subplots(figsize=(10, 4))
    for device, group in df.groupby("device", sort=True):
        group = group.sort_values("parsed_night_date")
        ax.plot(group["parsed_night_date"], group[metric], marker="o", label=device)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_xlabel("Date")
    ax.grid(True, alpha=0.25)
    ax.legend(loc="best")
    fig.autofmt_xdate()
    fig.tight_layout()
    return fig, ax


def plot_observation_metric(observations_df, metric, ylabel, title, device=None):
    df = observations_df.copy()
    if device is not None:
        df = df[df["device"].eq(device)]
    df = df[df["metric"].eq(metric)]
    if df.empty:
        return None
    df["parsed_night_date"] = pd.to_datetime(df["night_date"], errors="coerce")
    df["numeric_value"] = pd.to_numeric(df["value"], errors="coerce")
    df = df.dropna(subset=["parsed_night_date", "numeric_value"])
    if df.empty:
        return None

    fig, ax = plt.subplots(figsize=(10, 4))
    for source_file, group in df.groupby("source_file", sort=True):
        group = group.sort_values("parsed_night_date")
        ax.plot(group["parsed_night_date"], group["numeric_value"], marker="o", label=source_file)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_xlabel("Date")
    ax.grid(True, alpha=0.25)
    ax.legend(loc="best")
    fig.autofmt_xdate()
    fig.tight_layout()
    return fig, ax


In [ ]:
available_metrics = available_numeric_metrics(nightly_summary)
for metric in PLOT_METRICS:
    if metric in OPTIONAL_PLOT_METRICS and metric not in available_metrics:
        print(f"No CPAP metrics detected; skipping optional {metric} trend.")
        continue
    ylabel, title = PLOT_SPECS.get(metric, (CANONICAL_METRIC_UNITS.get(metric, "Value"), metric))
    plot_metric(nightly_summary, metric, ylabel, title)

for metric, (ylabel, title) in MINDMONITOR_TREND_SPECS.items():
    if metric in set(observations["metric"].dropna()):
        plot_observation_metric(observations, metric, ylabel, title, device=MINDMONITOR_DEVICE)
plt.show()


## Extraction Report

In [ ]:
if report_path.exists():
    print(report_path.read_text(encoding="utf-8"))
else:
    print("No extraction report found yet.")

## TODO: Remaining structured imports

MindMonitor CSV sensor logs now have first-pass ingestion. Remaining native export importers should return the same long-form observation schema used by `device_observations_long.csv`.


In [ ]:
# TODO: Add Oura CSV/JSON import for Ring 4 finger and Ring 3 toe exports.
def load_oura_export(path):
    raise NotImplementedError("Parse Oura native exports into normalized observations.")


# TODO: Add Samsung Health / SleepWatch import.
def load_samsung_sleepwatch_export(path):
    raise NotImplementedError("Parse Samsung/SleepWatch native exports into normalized observations.")


# TODO: Add Muse screenshot/PDF table or native export import beyond MindMonitor sensor CSVs.
def load_muse_export(path):
    raise NotImplementedError("Parse Muse exports into normalized observations.")


# TODO: Add OSCAR and SleepScope structured export import.
def load_cpap_export(path):
    raise NotImplementedError("Parse OSCAR/SleepScope exports into normalized observations.")
